# rearrange-as-sequential-layer — worked example 1: Conv to flatten to Linear with Rearrange

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rearrange-as-sequential-layer`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
from einops.layers.torch import Rearrange

## Concept

`einops.layers.torch.Rearrange` is the Module form of the rearrange operation, usable as a stateless layer inside `nn.Sequential`. Dropping it between a conv stack and a Linear flattens the feature map without writing a custom `forward`.

## Worked solution

We build `conv_classifier(in_channels, height, width, num_classes)` as a single `nn.Sequential`. A `Conv2d` with `padding=1` keeps the spatial size, a `ReLU` adds nonlinearity, then `Rearrange('b c h w -> b (c h w)')` flattens channels and spatial dims into one feature axis while preserving the batch dim, and finally a `Linear` maps to the class logits. We import the capital-R `Rearrange` from `einops.layers.torch` because it is the Module, not the function. We seed, build the model, push a batch through, and print the output shape, confirming `(B, num_classes)` with no custom forward written.

In [ ]:
import torch as t
from einops.layers.torch import Rearrange

t.manual_seed(0)

def conv_classifier(in_channels, height, width, num_classes):
    return t.nn.Sequential(
        t.nn.Conv2d(in_channels, 8, kernel_size=3, padding=1),
        t.nn.ReLU(),
        Rearrange('b c h w -> b (c h w)'),
        t.nn.Linear(8 * height * width, num_classes),
    )

model = conv_classifier(3, 8, 8, 5)
out = model(t.randn(4, 3, 8, 8))
print('output shape:', tuple(out.shape))